# 疑似ラベル（pseudo-labeling）・半教師あり

テスト予測を仮の正解として学習に加える疑似ラベルの手順と、リークを避ける注意点を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/advanced/03-pseudo-labeling`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# 初期学習用の訓練データと未ラベルのテストデータを用意します（検証セットは別途分離して保持）
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X_all, y_all = make_classification(n_samples=700, n_features=8, n_informative=5, random_state=42)
X_labeled, X_test, y_labeled, _ = train_test_split(X_all, y_all, test_size=250, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_labeled, y_labeled, test_size=0.25, stratify=y_labeled, random_state=42
)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
base_auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])
print(f"Baseline validation AUC (before pseudo-labeling): {base_auc:.4f}")

## 疑似ラベルとは

疑似ラベリングは、学習済みモデルが**テストデータに付けた予測**を「仮の正解」として
訓練データに加え、再学習する半教師あり的なテクニックです。ラベルなしデータを活用できます。

## 基本の手順

```text
1. 訓練データでモデルを学習し、テストを予測
2. 自信の高い予測（確率が極端に高い/低い）だけを疑似ラベルとして採用
3. 訓練データ + 疑似ラベルで再学習
4. 必要なら 1〜3 を数回繰り返す
```

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
confident = (proba > 0.95) | (proba < 0.05)        # 自信の高いものだけ
pseudo_X = X_test[confident]
pseudo_y = (proba[confident] > 0.5).astype(int)

X_aug = np.vstack([X_train, pseudo_X])
y_aug = np.concatenate([y_train, pseudo_y])
model.fit(X_aug, y_aug)

In [ ]:
after_auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])
print(f"Selected high-confidence pseudo-labels: {len(pseudo_X)} / {len(X_test)}")
print(f"Validation AUC (evaluated strictly on clean X_valid): {after_auc:.4f}")

## リークと過学習に注意

> **Warning:** 疑似ラベルは諸刃の剣です。**誤った予測を正解として学習**すると誤りが増幅し、 過学習やシェイクアップを招きます。検証は**本物のラベルだけ**で行い、 疑似ラベルを検証側に混ぜない（データリークを作らない）ことが鉄則です。

- 自信の高い予測だけを使う（閾値を厳しめに）
- 交差検証で「疑似ラベルあり/なし」を比較し、効いたときだけ採用
- fold ごとに疑似ラベルを作り直す実装にすると、よりリークを避けられる

> **Tip:** 疑似ラベルは、ラベルが少ない/テストが大きいコンペで効きやすい一方、効かないことも多い手法です。 「まず試す」より「ベースラインが固まってから慎重に試す」のが安全です。

## 次のステップ

次は、評価指標に合わせた後処理（閾値・順位最適化）を学びます。